In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [7]:
!pip -q install transformers datasets peft accelerate

In [8]:
import pandas as pd
import torch
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

from peft import LoraConfig, TaskType, get_peft_model

In [9]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for f in filenames:
        print("   ", f)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/smart-mcq-solver-challenge
    sample_submission.csv
    train.csv
    test.csv


In [10]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [11]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train["label"] = train["answer"].map(label_map)

print("Q1 =", train.loc[150,"label"])

Q1 = 2


In [12]:
print(train.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'label']


In [13]:
formatted = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])

print("Q2 =", len(formatted))

Q2 = 407


In [14]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

choices = []

for op in ["A", "B", "C", "D", "E"]:
    choices.append(
        str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, op])
    )

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = enc["input_ids"].unsqueeze(0)

print(input_ids.shape)
print("Q3 =", input_ids.shape[1])

torch.Size([1, 5, 128])
Q3 = 5


In [16]:
all_input_ids = []

for i in range(16):

    choices = []

    for op in ["A", "B", "C", "D", "E"]:
        choices.append(
            str(train.loc[i, "prompt"]) + " [SEP] " + str(train.loc[i, op])
        )

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    all_input_ids.append(enc["input_ids"])

all_input_ids = torch.stack(all_input_ids)

print("Shape:", all_input_ids.shape)

print("Q4 =", all_input_ids.numel())

Shape: torch.Size([16, 5, 128])
Q4 = 10240


In [17]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

choices = []

for op in ["A", "B", "C", "D", "E"]:
    choices.append(
        str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, op])
    )

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

outputs = model(
    input_ids=enc["input_ids"].unsqueeze(0),
    attention_mask=enc["attention_mask"].unsqueeze(0)
)

print("Logits Shape:", outputs.logits.shape)
print("Q5 =", outputs.logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits Shape: torch.Size([1, 5])
Q5 = 5


In [18]:
labels = torch.tensor([train.loc[0, "label"]])

outputs = model(
    input_ids=enc["input_ids"].unsqueeze(0),
    attention_mask=enc["attention_mask"].unsqueeze(0),
    labels=labels
)

print("Loss:", outputs.loss)
print("Q6 =", outputs.loss.dim())

Loss: tensor(1.6129, grad_fn=<NllLossBackward0>)
Q6 = 0


In [19]:
!pip uninstall -y torchao
!pip install -q torchao==0.16.0 peft --upgrade

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.5 MB/s eta 0:00:00:00:01


In [20]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Q7 =", trainable)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Failed to load /usr/local/lib/python3.12/dist

Q7 = 295681


In [21]:
from datasets import Dataset

dataset = []

for _, row in train.head(100).iterrows():

    choices = [
        str(row["prompt"]) + " [SEP] " + str(row[c])
        for c in ["A", "B", "C", "D", "E"]
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    dataset.append({
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": int(row["label"])
    })

hf_dataset = Dataset.from_list(dataset)

print("Shape:", np.array(hf_dataset[0]["input_ids"]).shape)
print("Q8 =", len(hf_dataset[0]["input_ids"]))

Shape: (5, 128)
Q8 = 5


In [22]:
from transformers import DefaultDataCollator

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="tmp",
        max_steps=4,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        report_to="none"
    ),
    train_dataset=hf_dataset.select(range(32)),
    data_collator=DefaultDataCollator()
)

trainer.train()

print("Q9 =", trainer.state.global_step)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Q9 = 4


In [23]:
row = train.iloc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[c])
    for c in ["A", "B", "C", "D", "E"]
]

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

with torch.no_grad():
    out = model(
        input_ids=enc["input_ids"].unsqueeze(0),
        attention_mask=enc["attention_mask"].unsqueeze(0)
    )

prob = torch.softmax(out.logits, dim=1)

print(prob)
print("Q10 =", round(prob[0,4].item(),4))

tensor([[0.2032, 0.2022, 0.2080, 0.1987, 0.1880]])
Q10 = 0.188
